# ３章 データ構造 

## 3.1 Pythonのネイティブなデータ構造

### 3.1.1 リスト

In [28]:
small_list = list(range(10))

In [29]:
%%timeit
last_element = small_list[-1]

9.53 ns ± 0.0226 ns per loop (mean ± std. dev. of 7 runs, 100,000,000 loops each)


In [30]:
large_list = list(range(10_000))

In [31]:
%%timeit
last_element = large_list[-1]

9.48 ns ± 0.019 ns per loop (mean ± std. dev. of 7 runs, 100,000,000 loops each)


In [32]:
%%timeit
4200 in small_list   # 4200が small_listにあるか

38.9 ns ± 0.0498 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


In [33]:
%%timeit
4200 in large_list

14.3 μs ± 8.56 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


### 3.1.3 辞書

##### メモ： 「`pip install Faker`」でインストール

In [34]:
from faker import Faker

fake = Faker()

In [35]:
small_dict = {}
for i in range(10):
    small_dict[fake.name()] = fake.address()
name = list(small_dict.keys())[0]  # 先頭要素のキーを得る（ここでは0でなくても構わない）    

In [36]:
name

'Philip Wilson'

In [37]:
%%timeit
small_dict[name]

9.21 ns ± 0.0403 ns per loop (mean ± std. dev. of 7 runs, 100,000,000 loops each)


In [38]:
large_dict = {}
for i in range(10_000):
    large_dict[fake.name()] = fake.address()
name = list(large_dict.keys())[0]

In [39]:
%%timeit
large_dict[name]

9.25 ns ± 0.025 ns per loop (mean ± std. dev. of 7 runs, 100,000,000 loops each)


### 3.1.4 セット（集合）

In [40]:
%%timeit
4200 in large_list

14.3 μs ± 20.7 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [41]:
%%timeit
large_set = set(large_list)
4200 in large_set

46.4 μs ± 145 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [42]:
large_set = set(large_list)

In [43]:
%%timeit
4592 in large_set

11.2 ns ± 0.0303 ns per loop (mean ± std. dev. of 7 runs, 100,000,000 loops each)


## 3.2 numPy

### 3.2.1　NumPyの関数

In [44]:
python_2d_list =[[1, 3, 5], [2, 4, 6], [7, 9, 11]]

In [45]:
first_column = [python_2d_list[i][0] for i in range(len(python_2d_list))]

In [46]:
first_column

[1, 2, 7]

In [47]:
import numpy as np

np_2d_array = np.array([[1, 3, 5], [2, 4, 6], [7, 9, 11]])

first_column = np_2d_array[:, 0]

In [48]:
first_column

array([1, 2, 7])

### 3.2.2　NumPyのパフォーマンスに関する考察

In [49]:
mixed_type_list = ["one", 2, 3.14]

In [50]:
mixed_type_array = np.array(["one", 2, 3.14])

In [51]:
print(mixed_type_array)

['one' '2' '3.14']


In [52]:
integer_array = np.array([1, 2, 3])
print(integer_array.dtype)

int64


##### ●Pythonのリストと比較して、NumPyを使うことでパフォーマンスが向上する例

In [53]:
random_int_array = np.random.randint(1, 100_000, 100_000)
random_int_list = list(random_int_array)

In [54]:
%%timeit -r 7 -n 100
sum(random_int_list)

1.3 ms ± 49.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [55]:
%%timeit -r 7 -n 100
np.sum(random_int_array)


9.22 μs ± 1.39 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


##### ●配列を適切なスペースで初期化する

In [56]:
array_to_fill = np.zeros(1000)

In [57]:
random_int_array = np.random.randint(1, 100_000, 100_000)

In [58]:
random_int_array.nbytes

800000

In [59]:
random_int_array.dtype

dtype('int64')

In [60]:
random_int_array_32 = random_int_array.astype(np.int32)

In [61]:
random_int_array_32.nbytes

400000

In [62]:
small_array = np.array([1, 3, 5], dtype=np.int16)

In [63]:
small_array.dtype

dtype('int16')

### 3.2.3　Daskを使った配列演算

##### ●環境によってはすごく時間がかかるかもしれないので、注意。30億 -> 10億でよいかも

In [64]:
large_np_array = np.random.randint(1, 100000, 30_0000_0000)

In [65]:
%%timeit -r 1 -n 7
np.max(large_np_array)

7.21 s ± 0 ns per loop (mean ± std. dev. of 1 run, 7 loops each)


In [66]:
import dask.array as da

large_np_array = np.random.randint(1, 100000, 30_0000_0000)

In [67]:
large_dask_array = da.from_array(large_np_array)

In [68]:
%%timeit -r 1 -n 7
array_max = large_dask_array.max()
array_max.compute()

3.41 s ± 0 ns per loop (mean ± std. dev. of 1 run, 7 loops each)


In [69]:
from dask.distributed import Client

client = Client(n_workers=4)

2025-07-29 23:34:32,884 - tornado.application - ERROR - Exception in callback <bound method SystemMonitor.update of <SystemMonitor: cpu: 1 memory: 58 MB fds: 162>>
Traceback (most recent call last):
  File "/opt/homebrew/lib/python3.11/site-packages/tornado/ioloop.py", line 937, in _run
    val = self.callback()
          ^^^^^^^^^^^^^^^
  File "/opt/homebrew/lib/python3.11/site-packages/distributed/system_monitor.py", line 168, in update
    net_ioc = psutil.net_io_counters()
              ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/lib/python3.11/site-packages/psutil/__init__.py", line 2148, in net_io_counters
    rawdict = _psplatform.net_io_counters()
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 12] Cannot allocate memory


In [70]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 16,Total memory: 24.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:61641,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:61652,Total threads: 4
Dashboard: http://127.0.0.1:61656/status,Memory: 6.00 GiB
Nanny: tcp://127.0.0.1:61644,


### 3.3 機械学習における配列

In [71]:
import numpy as np

np_tensor = np.random.rand(4,4)

In [72]:
np_tensor

array([[0.04567769, 0.79427977, 0.1498245 , 0.48649097],
       [0.79640863, 0.10995152, 0.67051172, 0.63342842],
       [0.81268441, 0.60003291, 0.19625202, 0.90805349],
       [0.51302668, 0.1266429 , 0.86710009, 0.12753921]])

##### メモ： 「`pip install tensorflow`」でインストール

In [73]:
import tensorflow as tf

tf_tensor = tf.convert_to_tensor(np_tensor)

In [74]:
tf_tensor

<tf.Tensor: shape=(4, 4), dtype=float64, numpy=
array([[0.04567769, 0.79427977, 0.1498245 , 0.48649097],
       [0.79640863, 0.10995152, 0.67051172, 0.63342842],
       [0.81268441, 0.60003291, 0.19625202, 0.90805349],
       [0.51302668, 0.1266429 , 0.86710009, 0.12753921]])>

##### メモ： 「`pip install torch`」でインストール

In [75]:
import torch

pytorch_tensor = torch.from_numpy(np_tensor)

## 3.4 pandas

### 3.4.1 DataFrame の機能

In [77]:
import pandas as pd
usa_data = pd.Series([13.33, 14.02, 14.02, 14.25],
                     index=["2000", "2001", "2002", "2003"])

In [78]:
usa_data

2000    13.33
2001    14.02
2002    14.02
2003    14.25
dtype: float64

In [80]:
india_data = pd.Series([9.02, 9.01, 8.84, 8.84],
                       index=["2000", "2001", "2002", "2003"])
df = pd.DataFrame({'USA': usa_data, 'India': india_data})

In [81]:
df

,USA,India
2000,13.33,9.02
2001,14.02,9.01
2002,14.02,8.84
2003,14.25,8.84


In [84]:
%%timeit
df["India_fraction"] = df["India"] / 100

21.3 μs ± 87.5 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [86]:
%%timeit
df["India_fraction"] = df["India"].apply(lambda x: x / 100)

21.5 μs ± 277 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [87]:
%%timeit
df["India_fraction"] = [row['India'] / 100 for index, row in df.iterrows()]

40.1 μs ± 328 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
